# Contest inference profile (EVA02 extract cycle)

Organizer latency/FPS is **not** a raw `model()` microbench and **not** retrieval.

Timed: one vehicle through `extract()` — disk read, decode, bbox crop, preprocess, H2D, forward, postprocess, L2. Untimed: gallery search, re-ranking, AQE/DBA (those scale with gallery size).

| Metric | Protocol |
| --- | --- |
| `latency_b1` | median of 300 full cycles at batch=1 after 50 warmups; CUDA sync before and after every timed sample |
| `throughput` | sustained images/s at batches 1 / 8 / 16 / 32, each window ≥ 10 s; score is the best FPS |
| Weights | sum of `.pt .pth .bin .onnx .engine .plan .safetensors .ckpt .trt .pb .tflite .npz` under the solution dirs; cap **2 GiB** |
| Also logged | peak VRAM, weight-load time (reference), two-run determinism |
| Performance | `10% × latency_score + 10% × throughput_score`. ≤40 ms and ≥100 FPS are full marks; >80 ms or <50 FPS are zero. Files over 2 GiB are not admitted. |

A Lightning training `.ckpt` (optimizer, loops, raw+EMA) is not the contest payload. Export EMA with `scripts/export_serving.py` to `weights/finetuned/eva02.pt`. CatBoost `.cbm` is listed here but is **outside** the official suffix glob.

Set `PROFILE_FAST=1` to shorten warmup/repeats/windows. Default is the contest protocol.


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hydra import compose, initialize_config_dir

from augmentations import build_transforms
from dataset.images import image_path
from eval import load_model
from profiling import (
    CONTEST_BATCH_SIZES,
    CONTEST_MIN_SECONDS,
    CONTEST_REPEATS,
    CONTEST_WARMUP,
    LIMIT_BYTES,
    STAGES,
    as_jsonable,
    bytes_text,
    inventory,
    performance_scores,
    pick_device,
    profile_extract,
    time_load,
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)


def display(obj):
    print(obj if isinstance(obj, str) else obj.to_string() if hasattr(obj, "to_string") else obj)


CV = ROOT / "runs" / "cv" / "eva02_trial23"
FOLD = 0
VAL = CV / f"fold{FOLD}" / "val"
SERVE = ROOT / "weights" / "finetuned" / "eva02.pt"
FIG_DIR = ROOT / "notebooks" / "eva02" / "inference_profile_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
FAST = os.environ.get("PROFILE_FAST") == "1"
WARMUP = 5 if FAST else CONTEST_WARMUP
REPEATS = 20 if FAST else CONTEST_REPEATS
MIN_SECONDS = 1.0 if FAST else CONTEST_MIN_SECONDS
BATCH_SIZES = (1, 8) if FAST else CONTEST_BATCH_SIZES
STAGE_REPEATS = 4 if FAST else 16
DEVICE = pick_device(prefer=(2,), exclude=(0, 1))
print("ROOT", ROOT)
print("device", DEVICE)
print(
    "protocol",
    {
        "warmup": WARMUP,
        "repeats": REPEATS,
        "min_seconds": MIN_SECONDS,
        "batch_sizes": BATCH_SIZES,
        "fast": FAST,
    },
)

Please 'pip install xformers'
Please 'pip install xformers'
ROOT /data/projects/ryzhichkin/ReID
device cuda:2
protocol {'warmup': 50, 'repeats': 300, 'min_seconds': 10.0, 'batch_sizes': (1, 8, 16, 32), 'fast': False}
/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error timed out>
  data = fetch_version_info()
/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Weight inventory vs 2 GiB

Scan `weights/finetuned/` only. That is the contest payload (`eva02.pt`). The training Lightning `.ckpt` is logged as a comparison and is **not** submitted.


In [4]:
metrics_path = VAL / "metrics.json"
ckpt_path = Path(json.loads(metrics_path.read_text())["checkpoint"]) if metrics_path.is_file() else None
roots = [ROOT / "weights" / "finetuned"]
weights = inventory(roots)
files = pd.DataFrame(weights["files"] + weights["extra_files"])
if len(files):
    files["mib"] = files["bytes"] / 1024**2
    files["official"] = files["suffix"].isin(set(item["suffix"] for item in weights["files"]))
    display(
        files.sort_values("bytes", ascending=False)[["name", "suffix", "bytes", "mib", "official", "path"]]
    )
else:
    print("no weight files yet under", roots)
print(
    "official",
    bytes_text(weights["total_bytes"]),
    "limit",
    bytes_text(weights["limit_bytes"]),
    "headroom",
    bytes_text(weights["headroom_bytes"]),
    "within_2gib",
    weights["within_limit"],
    "extra_unscanned",
    bytes_text(weights["extra_bytes"]),
)
if ckpt_path is not None and ckpt_path.is_file():
    print("training lightning ckpt (not submitted)", ckpt_path, bytes_text(ckpt_path.stat().st_size))

       name suffix       bytes       mib  official                                                       path
0  eva02.pt    .pt  1228952319 1172.0203      True  /data/projects/ryzhichkin/ReID/weights/finetuned/eva02.pt
official 1.14 GiB limit 2.00 GiB headroom 875.98 MiB within_2gib True extra_unscanned 0 B
training lightning ckpt (not submitted) /data/projects/ryzhichkin/ReID/runs/cv/eva02_trial23/fold0/checkpoints/epoch025.ckpt 4.55 GiB


## Load serving model

Hydra `experiment=current_best_tuned`, `weights/finetuned/eva02.pt` (exported EMA), `eval.precision=bf16`, TTA off. Device picker skips GPUs 0–1 and prefers `cuda:2` when it is ≥95% free.


In [6]:
REPORT = None
MODEL = None
CFG = None
if not SERVE.is_file():
    print("skip load: weights/finetuned/eva02.pt missing")
elif DEVICE.type != "cuda":
    print("skip load: no free CUDA device (refusing CPU EVA02-L)")
else:
    with initialize_config_dir(version_base="1.3", config_dir=str(ROOT / "configs")):
        hydra_cfg = compose(
            config_name="config",
            overrides=[
                f"checkpoint={SERVE}",
                "experiment=current_best_tuned",
                f"eval.device={DEVICE}",
                "eval.precision=bf16",
                "eval.weights=auto",
            ],
        )

    def _load():
        model, cfg, checkpoint, choice = load_model(hydra_cfg, override_items=[])
        model.to(DEVICE).eval()
        return model, cfg, checkpoint, choice

    packed, load_ms, load_snap = time_load(_load, DEVICE)
    MODEL, CFG, CHECKPOINT, CHOICE = packed
    print(
        "weights",
        CHOICE,
        "load",
        f"{load_ms:.0f} ms",
        "alloc",
        bytes_text(load_snap["allocated_bytes"]),
        "peak",
        bytes_text(load_snap["peak_bytes"]),
    )
    print(
        "pooling",
        CFG.model.pooling.kind,
        "size",
        list(CFG.data.image_size),
        "context",
        float(CFG.data.context_pct),
        "precision",
        CFG.eval.precision,
        "tta",
        bool(CFG.eval.tta.enabled),
    )

weights ema load 4144 ms alloc 1.14 GiB peak 1.14 GiB
pooling attn size [336, 336] context 6.201741080661072 precision bf16 tta False


## Run extract profile

Sample the fold-0 val query list (file order). Gallery search is not in this loop.


In [8]:
if MODEL is None or CFG is None:
    print("skip profile")
else:
    query = pd.read_csv(VAL / "query.csv", dtype={"image_id": str})
    n = max(BATCH_SIZES)
    sample = query.head(n).copy()
    if len(sample) < n:
        raise ValueError(f"need {n} query rows, got {len(sample)}")
    paths = [image_path(ROOT / "data" / "images", image_id) for image_id in sample.image_id]
    bboxes = sample[["x", "y", "w", "h"]].to_numpy().tolist()
    transform = build_transforms(CFG, train=False)
    REPORT = profile_extract(
        roots=roots,
        paths=paths,
        bboxes=bboxes,
        transform=transform,
        model=MODEL,
        device=DEVICE,
        load_ms=load_ms,
        checkpoint=CHECKPOINT,
        weights=CHOICE,
        context_pct=float(CFG.data.context_pct),
        precision=str(CFG.eval.precision),
        tta=CFG.eval.tta,
        model_cfg=CFG.model,
        warmup=WARMUP,
        repeats=REPEATS,
        batch_sizes=BATCH_SIZES,
        min_seconds=MIN_SECONDS,
        vram_repeats=2,
        stage_repeats=STAGE_REPEATS,
    )
    (FIG_DIR / "report.json").write_text(json.dumps(as_jsonable(REPORT), indent=2))
    c = REPORT["contest"]
    board = pd.DataFrame(
        [
            {"metric": "latency_b1_ms", "value": c["latency_b1_ms"], "note": "median full cycle, batch=1"},
            {
                "metric": "throughput_best_fps",
                "value": c["throughput_best_fps"],
                "note": f"best at batch {c['throughput_best_batch_size']}",
            },
            {"metric": "peak_vram", "value": c["peak_vram_bytes"], "note": bytes_text(c["peak_vram_bytes"])},
            {"metric": "load_ms", "value": c["load_ms"], "note": "reference, not scored"},
            {
                "metric": "official_weight_bytes",
                "value": c["weight_bytes"],
                "note": bytes_text(c["weight_bytes"]),
            },
            {
                "metric": "serving_tensor_bytes",
                "value": c["serving_tensor_bytes"],
                "note": bytes_text(c["serving_tensor_bytes"]),
            },
            {"metric": "within_2gib", "value": float(c["within_2gib"]), "note": bytes_text(LIMIT_BYTES)},
            {
                "metric": "performance_eligible",
                "value": float(c["performance_eligible"]),
                "note": "official weight files vs 2 GiB",
            },
            {"metric": "latency_score", "value": c["latency_score"], "note": "1 if <=40 ms, 0 if >80 ms"},
            {
                "metric": "throughput_score",
                "value": c["throughput_score"],
                "note": "1 if >=100 FPS, 0 if <50",
            },
            {
                "metric": "performance_score",
                "value": c["performance_score"],
                "note": "0.10*lat + 0.10*fps (max 0.20)",
            },
            {"metric": "deterministic", "value": float(c["deterministic"]), "note": REPORT["determinism"]},
        ]
    )
    display(board)
    ema_ok = c["serving_tensor_bytes"] <= LIMIT_BYTES
    print(
        "EMA-only performance",
        performance_scores(c["latency_b1_ms"], c["throughput_best_fps"], within_limit=ema_ok),
    )
    display(pd.DataFrame([REPORT["stages_per_image_ms"]]))
    display(pd.DataFrame(REPORT["throughput"]["by_batch"]))
    display(pd.DataFrame(REPORT["vram"]))
    print("determinism", REPORT["determinism"])
    if REPORT["checkpoint"] is not None:
        keys = REPORT["checkpoint"]["bytes_by_key"]
        display(
            pd.DataFrame(
                {
                    "key": list(keys),
                    "bytes": list(keys.values()),
                    "text": [bytes_text(v) for v in keys.values()],
                }
            )
            .sort_values("bytes", ascending=False)
            .head(12)
        )

                  metric           value                                                                                                                                   note
0          latency_b1_ms         31.2240                                                                                                             median full cycle, batch=1
1    throughput_best_fps         65.6559                                                                                                                       best at batch 32
2              peak_vram 4083816960.0000                                                                                                                               3.80 GiB
3                load_ms       4144.1160                                                                                                                  reference, not scored
4  official_weight_bytes 1228952319.0000                                                                                

## Stage costs (batch=1, CUDA-synced)

Each bar syncs CUDA before and after that stage, so the sum is **larger** than contest `latency_b1` (one sync around the whole cycle). Use this chart for *where* time goes, not as the scored number.


In [10]:
if REPORT is None:
    print("skip figure")
else:
    stages = REPORT["stages_per_image_ms"]
    fig, ax = plt.subplots()
    names = list(STAGES)
    vals = [stages[name] for name in names]
    ax.bar(names, vals, color="#3b6ea5")
    ax.set_ylabel("ms / image")
    ax.set_title("extract() stage breakdown (mean of timed batch=1)")
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    fig.savefig(FIG_DIR / "stages.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    print("wrote", FIG_DIR / "stages.png")

wrote /data/projects/ryzhichkin/ReID/notebooks/eva02/inference_profile_figs/stages.png


![stages](inference_profile_figs/stages.png)


## latency_b1 distribution


In [13]:
if REPORT is None:
    print("skip figure")
else:
    samples = np.asarray(REPORT["latency"]["samples_ms"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(samples, bins=min(40, max(8, len(samples) // 5)), color="#3b6ea5", edgecolor="white")
    axes[0].axvline(
        REPORT["latency"]["p50_ms"], color="#c0392b", label=f"p50 {REPORT['latency']['p50_ms']:.2f} ms"
    )
    axes[0].axvline(
        REPORT["latency"]["p90_ms"], color="#d68910", label=f"p90 {REPORT['latency']['p90_ms']:.2f} ms"
    )
    axes[0].set_xlabel("ms")
    axes[0].set_ylabel("runs")
    axes[0].set_title("batch=1 full-cycle latency")
    axes[0].legend()
    axes[1].boxplot(samples, vert=True, widths=0.5)
    axes[1].set_ylabel("ms")
    axes[1].set_title("latency_b1 spread")
    fig.savefig(FIG_DIR / "latency.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    print(
        "p50",
        REPORT["latency"]["p50_ms"],
        "p90",
        REPORT["latency"]["p90_ms"],
        "p99",
        REPORT["latency"]["p99_ms"],
        "mean",
        REPORT["latency"]["mean_ms"],
    )

p50 31.224003992974758 p90 33.99848714470863 p99 36.76809821277856 mean 31.67141641800602
notebooks/eva02/inference_profile.ipynb:cell12:13: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.


![latency](inference_profile_figs/latency.png)


## Throughput vs batch size


In [16]:
if REPORT is None:
    print("skip figure")
else:
    rows = pd.DataFrame(REPORT["throughput"]["by_batch"])
    fig, ax = plt.subplots()
    ax.bar([str(v) for v in rows.batch_size], rows.fps, color="#1e8449")
    ax.set_xlabel("batch size")
    ax.set_ylabel("images / s")
    ax.set_title("sustained extract() FPS (best bar is the contest throughput score)")
    for i, fps in enumerate(rows.fps):
        ax.text(i, fps, f"{fps:.1f}", ha="center", va="bottom", fontsize=8)
    fig.savefig(FIG_DIR / "throughput.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    display(rows)

   batch_size  images  seconds     fps  ms_per_image
0           1     308  10.0121 30.7629       32.5067
1           8     608  10.1013 60.1900       16.6141
2          16     656  10.1154 64.8516       15.4198
3          32     672  10.2352 65.6559       15.2309


![throughput](inference_profile_figs/throughput.png)


## Peak VRAM vs batch size


In [19]:
if REPORT is None:
    print("skip figure")
else:
    rows = pd.DataFrame(REPORT["vram"])
    rows["peak_mib"] = rows.peak_bytes / 1024**2
    rows["alloc_mib"] = rows.allocated_bytes / 1024**2
    fig, ax = plt.subplots()
    x = np.arange(len(rows))
    ax.bar(x - 0.18, rows.peak_mib, 0.36, label="peak", color="#6c3483")
    ax.bar(x + 0.18, rows.alloc_mib, 0.36, label="allocated", color="#a569bd")
    ax.set_xticks(x, [str(v) for v in rows.batch_size])
    ax.set_xlabel("batch size")
    ax.set_ylabel("MiB")
    ax.set_title("CUDA memory after extract() (weights already loaded)")
    ax.legend()
    fig.savefig(FIG_DIR / "vram.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    display(rows[["batch_size", "peak_bytes", "allocated_bytes", "reserved_bytes", "peak_mib"]])

   batch_size  peak_bytes  allocated_bytes  reserved_bytes  peak_mib
0           1  1940524032       1256074752      4789895168 1850.6279
1           8  2404271104       1256074752      4789895168 2292.8916
2          16  2965339136       1256074752      4789895168 2827.9678
3          32  4083816960       1256074752      4789895168 3894.6313


![vram](inference_profile_figs/vram.png)


## Weight bytes vs 2 GiB cap


In [22]:
fig, ax = plt.subplots()
official = weights["total_bytes"]
serving = 0 if REPORT is None else REPORT["contest"]["serving_tensor_bytes"]
labels = ["official files", "EMA/raw tensors", "2 GiB cap"]
vals = [official / 1024**3, serving / 1024**3, LIMIT_BYTES / 1024**3]
colors = ["#c0392b" if official > LIMIT_BYTES else "#1e8449", "#3b6ea5", "#7f8c8d"]
ax.barh(labels, vals, color=colors)
ax.set_xlabel("GiB")
ax.set_title("submit weight budget")
for i, v in enumerate(vals):
    ax.text(v, i, f" {v:.2f} GiB", va="center")
fig.savefig(FIG_DIR / "weights.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(
    "official", bytes_text(official), "serving_tensors", bytes_text(serving), "cap", bytes_text(LIMIT_BYTES)
)

official 1.14 GiB serving_tensors 1.14 GiB cap 2.00 GiB


![weights](inference_profile_figs/weights.png)


## Takeaways

Fold-0 EVA02-L-14-336, EMA, bf16, TTA off, `cuda:2`, contest protocol (50 warmup + 300 timed, ≥10 s/batch):

| Metric | Value |
| --- | --- |
| `latency_b1` | **31.2 ms** (p90 33.9 ms) |
| `throughput` | **65.7 FPS** at batch 32 (batch 1 is 30.8 FPS) |
| Peak VRAM | 1.81 GiB @ b1 → **3.80 GiB @ b32** (weights already loaded) |
| Load | 4.1 s, 1.14 GiB allocated |
| Serving `eva02.pt` | **~1.14 GiB** (EMA tensors + cfg; under the 2 GiB cap) |
| Training Lightning `.ckpt` | 4.55 GiB (optimizer ~2.27 GiB; **not submitted**) |
| Determinism | bit-identical on two runs |
| Official performance | latency_score **1.0**, throughput_score **0.313**, `performance_score` **0.131 / 0.20** |

- Scoring uses **full extract()**, not isolated forward. Decode (~9 ms) and preprocess (~3 ms) show up in `latency_b1`; gallery search and rerank do not.
- Submit `weights/finetuned/eva02.pt` from `scripts/export_serving.py`. Do not copy the training Lightning dump into the image or the run is excluded from the 20% performance score.
- Best FPS is 65.7, so throughput is on the 50–100 linear ramp (need 100 for a full 10%). Latency already has the full 10% (31.2 ms ≤ 40 ms).
- `.cbm` refusal heads are listed here but are **outside** the official suffix glob.
- Per-stage bars overstate GPU time because they sync between stages; `latency_b1` is the number that matches the harness.
